In [4]:
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import corner
import asdf
from tqdm import tqdm
import subprocess

# Python interpreter path: /nvme/scratch/software/anaconda3/envs/lewi_galfind/bin/python

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


GALAXY_ID = TABLE["SURVEY_ID"]        # object ID, used for labeling/output files
SERSIC_FILTERS = ["F444W","F356W","F277W"]                    # filter name, used for labeling/output files
SURVEY = TABLE["SURVEY"]

CUTOUT_SIZE = 0.30 #as
PIXEL_SIZE = 0.03 #as

In [5]:
def run_fits(wanted):
    """
    Runs morphometrica fits on three filters for each of the galaxies.

    Parameters
    ----------
    wanted : list of str
        List of parameters I want to extract from the fit results.

    Returns
    -------
    table : astropy.table
        Table of data with my wanted parameter results.
    """
    colnames = ["ID"]
    for filt in SERSIC_FILTERS:
        colnames.extend([f"{filt}_{param}" for param in wanted])

    rows = []

    for i in tqdm(range(len(GALAXY_ID)),total=len(GALAXY_ID)):
            
            row = {"ID": GALAXY_ID[i]}

            # Fill everything with NaN initially to make the length of the row match wanted
            for filt in SERSIC_FILTERS:
                for param in wanted:
                    row[f"{filt}_{param}"] = np.nan

            for filt in SERSIC_FILTERS:

                try:
                    # science_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_0.9as/{GALAXY_ID[i]}/{filt}.fits" for old 0.96 as
                    science_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{GALAXY_ID[i]}/cutouts/{GALAXY_ID[i]}_science_{filt}.fits"
                    # psf_path = f"/nvme/scratch/work/alberttg/Summer_project/PSFs/{SURVEY[i]}/{filt}_psf_norm.fits" for old 0.96 as
                    psf_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{GALAXY_ID[i]}/cutouts/{GALAXY_ID[i]}_psf_{filt}.fits"

                    result = subprocess.run(f"python /nvme/scratch/work/westcottl/Codes/Morfometryka/Code/morfometryka965.py {science_path} {psf_path} noshow rerun", 
                                   shell=True, capture_output=True, text=True)
                    
                    lines = result.stdout.splitlines()

                    header = None
                    values = None

                    for j, line in enumerate(lines):
                        if line.strip().startswith("# rootname9.65"):
                            header = [x.strip() for x in line.strip()[1:].split(",")]
                            values = [x.strip() for x in lines[j+1].split(",")]
                            break

                    if header is None:
                        continue

                    result_dict = dict(zip(header, values))

                    for param in wanted:
                        if param in result_dict:
                            try:
                                row[f"{filt}_{param}"] = float(result_dict[param])
                            except ValueError:
                                row[f"{filt}_{param}"] = result_dict[param]
                                # In case nan and cant make float
                
                except Exception as e:
                    print(e)
            rows.append(row)

    table = Table(rows=rows, names=colnames)
    table.write(f"Morphometrica_table_3.0as.csv", format="csv", overwrite=True)
    return table

In [6]:
if __name__ == "__main__":

    wanted = ["C1", "A0","S1", "G", "M20", "nFit2D", "QF"]

    table = run_fits(wanted)
    

  0%|          | 0/144 [00:00<?, ?it/s]

100%|██████████| 144/144 [43:50<00:00, 18.26s/it]
